In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np  
import pandas as pd

In [2]:
!pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable


In [3]:
import sys
print(sys.executable)


c:\Users\tanis\Desktop\Deep_learning_pjt\venv\python.exe


In [4]:
import sklearn
print(sklearn.__version__)


1.8.0


In [5]:
#loas the trained model,scaler,pickle,onehot
model=load_model('model.h5')

#load the encoder and scaler
with open('onehot_encoder_geo.pkl', 'rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:      
    scaler = pickle.load(file)

In [6]:
#Example input data
input_data = {
    'CreditScore':600,
    'Geography': 'France',      
    'Gender': 'Male',
    'Age': 40,          
    'Tenure': 3,
    'Balance': 60000, 
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [7]:
#one hot encode the geography
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
#label encode the gender
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head()

c:\Users\tanis\Desktop\Deep_learning_pjt\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [8]:
#combine the encoded features with the rest of the input data   
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [9]:
#Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender']) 
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [10]:
##Concation of one hot encoded
input_df = pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [11]:
#scaling the input data
input_scaled = scaler.transform(input_df)       
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [12]:
#predict churn
prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 413ms/step


array([[0.02872495]], dtype=float32)

In [13]:
prediction_proba=prediction[0][0]


In [14]:
prediction_proba

0.028724952

In [15]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")       
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
